In [1]:
import warnings
warnings.filterwarnings("ignore")
                        
import altair as alt
import folium
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2
import chart_utils

credentials, _ = google.auth.default()

In [2]:
sofi_trips = D1.filter_fct_daily_schedule_rt_route_direction_summary_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times  
)

In [15]:
sofi_stop_arrivals = D2.filter_fct_daily_scheduled_stops_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times,
)

In [ ]:
poi = gpd.read_parquet(
    f"{wc_vars.GCS_FILE_PATH}points_of_interest_{wc_vars.event_name}.parquet",
    storage_options = {"token": credentials.token},
    filters = [[("point_of_interest", "==", "SoFi Stadium")]]
)

TODO:
* find missing shapes (routes and stops dfs do not "speak" to each other, so arrivals will show up for LA Metro Bus, as barely any change, but LA Metro's special routes were under LA Metro Event feed.
* feed, route, and stop-name records can show change due to special events. comparing will take some manual and visual inspection to confirm we are comparing apples to apples, and not missing comparisons simply because strings are different.
* icon - limited set of icons, we can choose building, and change color, but perhaps we go with a buffer around the point itself? something more simple
* change route_name color (allow indiv routes to show up as a layer), but the one we want plotted should reflect service_change. more emphatic about service_changes seen in stops and routes.
   * see if we can use the same legend, clear up the busy-ness 

In [ ]:
m2 = sofi_trips[
    (sofi_trips.schedule_name.str.contains("LA Metro")) & 
    (sofi_trips.route_name.str.contains("14"))][
    ["schedule_name", "route_name", "direction_id", "geometry"]
].drop_duplicates().explore(
    "route_name",
    tiles = "CartoDB Positron",
    name = "route", legend=True,
    height=400, width=600
)

m2

In [ ]:
# set custom cmap and cutoffs example: PREDICTION_ERROR_COLORS, PREDICTION_ERROR_INDEX
# https://github.com/cal-itp/gtfs-curator/blob/main/rt_predictions/operator_report.ipynb
# change line width thickness example: sofi_service_changes.ipynb
import branca.colormap as cm

m = sofi_trips[
    ["schedule_name", "route_name", "direction_id", "geometry"]
].drop_duplicates().explore(
    "route_name",
    tiles = "CartoDB Positron",
    name = "route", legend=False
)
'''
m = poi.explore(
    "point_of_interest",
    m=m,
    marker_type = "marker",
    name="SoFi Stadium",
    marker_kwds=dict(icon=folium.DivIcon(class_name="mapIcon")),
    # can "fa fa-home" be switched to something else?
    style_kwds=dict(
        style_function=lambda x: {
            "html": f"""<span   class="fa fa-building" 
                                style="color:orange;
                                font-size:14px"></span>"""
        },
    ),
)
'''


folium.LayerControl().add_to(m)

m

In [ ]:
# https://stackoverflow.com/questions/73317052/geopandas-explore-how-to-set-marker-icon
# https://fontawesome.com/v4/icons/
# must take format "fa fa-[name_of_icon]"
poi.explore(
    "point_of_interest",
    marker_type = "marker",
    name="SoFi Stadium",
    marker_kwds=dict(icon=folium.DivIcon(class_name="mapIcon")),
    # can "fa fa-home" be switched to something else?
    style_kwds=dict(
        style_function=lambda x: {
            "html": f"""<span   class="fa fa-building" 
                                style="color:{x["properties"]["__folium_color"]};
                                font-size:14px"></span>"""
        },
    ),
)

In [ ]:
#<i class="fa-solid fa-stadium"></i>

## stop arrivals for time-of-day

In [2]:
sofi_stop_arrivals = D2.filter_fct_daily_scheduled_stops_to_special_routes_keep_far_stops(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times,
)

In [3]:
time_of_day = "pm_peak"
stop_arrivals_gdf = sofi_stop_arrivals

In [10]:
group_cols = ["schedule_name", "stop_id", "stop_name", "event_day", "day_type"]

# look at arrivals_per_hour_pm_peak for event, keep both day_types
event_df = stop_arrivals_gdf[
    (stop_arrivals_gdf.event_day == True) & 
    (stop_arrivals_gdf.event_time_of_day == time_of_day)
][
    ["service_date"] + group_cols + [f"arrivals_per_hour_{time_of_day}"]
].reset_index(drop=True)

# comparison is the same day-type, compare arrivals_per_hour_pm_peak
nonevent_df = stop_arrivals_gdf[
    (stop_arrivals_gdf.event_day == False) 
][["service_date"] + group_cols + [f"arrivals_per_hour_{time_of_day}"]].reset_index(drop=True)

# do something similar as arrivals_wide
time_of_day_df = pd.concat([event_df, nonevent_df], axis=0, ignore_index=True)

arrivals_by_event_df = D2.aggregate_by_event_type(
    time_of_day_df, 
    group_cols=group_cols,
    metric_cols = [f"arrivals_per_hour_{time_of_day}"]
)

In [11]:
arrivals_by_event_df = arrivals_by_event_df.assign(
    event_day=arrivals_by_event_df.event_day.map({True: "event", False: "non_event"})
)

In [12]:
df_wide = arrivals_by_event_df.pivot(
    index=["schedule_name", "stop_id", "stop_name"],
    columns=["day_type", "event_day"],
    values=[f"arrivals_per_hour_{time_of_day}"]
).reset_index()

In [13]:
# it is possible for there to be no weekend events
arrivals_by_event_df.groupby(["day_type", "event_day"]).agg({
    "schedule_name": "count"
})

schedule_name
day_type event_day               
weekday  event               2960
         non_event           3022
weekend  non_event           2703

In [14]:
df_wide.columns

MultiIndex([(            'schedule_name',        '',          ''),
            (                  'stop_id',        '',          ''),
            (                'stop_name',        '',          ''),
            ('arrivals_per_hour_pm_peak', 'weekday', 'non_event'),
            ('arrivals_per_hour_pm_peak', 'weekend', 'non_event'),
            ('arrivals_per_hour_pm_peak', 'weekday',     'event')],
           names=[None, 'day_type', 'event_day'])

In [15]:
# df_wide.columns.get_level_values(0)
df_wide.columns = [
    "_".join(col).rstrip("_").strip() for col in df_wide.columns.values
]

In [17]:
df_wide2 = df_wide.pipe(D2.merge_in_stop_geom, stop_arrivals_gdf)

In [21]:
def arrivals_for_time_of_day(
    stop_arrivals_gdf: gpd.GeoDataFrame, 
    time_of_day: str
):
    group_cols = ["schedule_name", "stop_id", "stop_name", "event_day", "day_type"]

    # look at arrivals_per_hour_pm_peak for event, keep both day_types
    event_df = stop_arrivals_gdf[
        (stop_arrivals_gdf.event_day == True) & 
        (stop_arrivals_gdf.event_time_of_day == time_of_day)
    ][
        ["service_date"] + group_cols + [f"arrivals_per_hour_{time_of_day}"]
    ].reset_index(drop=True)

    # comparison is the same day-type, compare arrivals_per_hour_pm_peak
    nonevent_df = stop_arrivals_gdf[
        (stop_arrivals_gdf.event_day == False) 
    ][["service_date"] + group_cols + [f"arrivals_per_hour_{time_of_day}"]].reset_index(drop=True)

    # do something similar as arrivals_wide
    time_of_day_df = pd.concat([event_df, nonevent_df], axis=0, ignore_index=True)
    
    arrivals_by_event_df = D2.aggregate_by_event_type(
        time_of_day_df, 
        group_cols=group_cols,
        metric_cols = [f"arrivals_per_hour_{time_of_day}"]
    )
    
    arrivals_wide = D2.make_wide(
        arrivals_by_event_df,
        index_cols=["schedule_name", "stop_id", "stop_name"],
        pivot_cols=["day_type", "event_day"],
        value_cols=[f"arrivals_per_hour_{time_of_day}"],
    ).pipe(merge_in_stop_geom, stop_arrivals_gdf)
    
    arrivals_wide = arrivals_wide.assign(
        combined_change = arrivals_wide[[
            f"change_arrivals_per_hour_{time_of_day}_weekday", 
            f"change_arrivals_per_hour_{time_of_day}_weekend"]].sum(axis=1)
    ).rename(columns = {"combined_change": f"combined_change_arrivals_per_hour_{time_of_day}"})

    return arrivals_wide
    

In [22]:
sofi_stop_arrivals_pm_peak = arrivals_for_time_of_day(sofi_stop_arrivals, "pm_peak")

KeyError: 'arrivals_per_hour_pm_peak_weekend_event'

In [ ]:
sofi_stop_arrivals_pm_peak.plot(
    "combined_change_arrivals_per_hour_pm_peak",
    cmap = "Blues", 
)

In [ ]:
sofi_stop_arrivals_pm_peak.explore(
    "combined_change_arrivals_per_hour_pm_peak",
    cmap = "Blues", tiles = "CartoDB Positron"
)

In [ ]:
sofi_stop_arrivals_pm_peak_full.explore(
    "combined_change_arrivals_per_hour_pm_peak",
    cmap = "Blues", tiles = "CartoDB Positron"
)

sofi_stop_arrivals_pm_peak_full.plot(
    "combined_change_arrivals_per_hour_pm_peak",
    cmap = "Blues",
)

## ideas
Contactless payments, before and after GIS map.
before event, more traffic
over time, how has that changed. more dynamic kind of chart, video?

heatmap, regression charts
* across dates, how it had changed, so it's little cubes, date on x-axis, routes on y-axis
* event on Saturday, compare to other saturdays (6 months or some period vs current Saturday with event)
* buffer ring analysis, near event, what changed 0-500meters, are there more trips, vs 500-1_000 meters, buffer to stop area, see changes to the stop. a buffer-ring analysis, splitting stops into distance bands from the stadium (say 0 to 500 meters, 500 meters to 1 kilometer, 1 to 2 kilometers), would show whether the service boost decays the further you get from the venue.

agency vs agency, specific agency that adds routes during event
mode related analysis, is it easier for bus routes to be added? or train frequencies?
date relative to event, google chart, event date is 0, and other dates are -1, 1, 2, 

bubble map, similar to ring map, but more suited for stops, can show change to bubble map for stops getting near stadiums

## Heatmap

In [ ]:
# heatmap
alt.Chart(sofi_trips).mark_rect().encode(
    y='route_name:O',
    x='service_date:T',
    color='sum(n_trips):Q'
)

In [ ]:
for i in arrivals_by_event_type.schedule_name.unique():
    one_chart = arrivals_by_operator_chart = alt.Chart(arrivals_by_event_type[arrivals_by_event_type.schedule_name==i]).mark_rect().encode(
        y='stop_name:O',
        x='event_day:O',
        color='sum(daily_arrivals):Q',
        tooltip=["sum(daily_arrivals)"]
    ).properties(title = i).interactive()
        
    display(one_chart)

## Bubble Chart by stop
* plot event-baseline (show the change in arrivals by stop)

## Time-of-day comparison


## Statistical Analysis

A control comparison can be used to assess how transit service changes on event days relative to typical service patterns. 

This fits in with comparing daily trips / daily stop arrivals for event vs non-event.
* weekend service is quite different than weekday, so without plotting them separately, we miss out the additional weekend service (since additional service still is much smaller than typical weekday service)

**Difference-in-difference or route-fixed effects**
* that event chart that shows before/event=0/after.
* comparison would be event day vs the last non-event day of similar type
   * Fri event, compared to Thurs (compare 2 weekdays)
   * Sat event, compare to last Sat event (compare 2 Saturdays, or compare to last Sunday?)
* **can compare event vs non-event, near vs far, bus vs rail all at once**!
   * should be able to tease out whether rail or bus added more service
   * think more about the set up for df. what happens if route name changes for special service? need to be able to add them onto the same record to compare what happened.
   * add these boolean columns (is_rail, is_near, is_event). remember to avoid perfect multicollinearity, always 1 less column
   * this might be able to take the entire df, don't need to filter for stops being within certain buffer. so we can take the entirety of those feeds, for all routes, for all stops, and just throw it into this regression, with the right dummy variables.

## Map of Routes for all other feeds
* map of routes (how to compare event vs non-event)?
   * aggregate daily trips on event days (weekday + weekend), aggregate daily trips on non-event days (weekday + weekend)
   * show the difference? 
* can we get stops that show up on routes as a layer too?
* see a chart that filters for that route, show weekday + weekend event vs non-event trips?
* can be combined with World Cup feed

In [ ]:
trips_by_event = sofi_trips.groupby(
    ["schedule_name", "event_day", "day_type", "route_name", "route_type"]
).agg({
    "n_trips": "sum",
    "service_date": "nunique"
}).reset_index().rename(columns = {
    "service_date": "n_days"
})

trips_by_event = trips_by_event.assign(
    daily_trips = trips_by_event.n_trips.divide(trips_by_event.n_days).round(1)
)

In [ ]:
trips_by_event_wide = D2.make_wide(
    trips_by_event,
    group_cols = ["schedule_name", "day_type", "route_name"],
    metric_cols = ["daily_trips"]
)

In [ ]:
alt.Chart(trips_by_event).mark_rect().encode(
    y='route_name:O',
    x='event_day:O',
    color='sum(daily_trips):Q'
)

In [ ]:
selection = alt.selection_point(fields=["schedule_name"], bind="legend")
alt.Chart(trips_by_event).mark_bar().encode(
    x="daily_trips",
    y=alt.Y("event_day", title = ""),
    row=alt.Row("day_type:N", title = ""),
    color=alt.Color("schedule_name:N"), 
    opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
).add_params(selection).transform_filter(selection)

In [ ]:
alt.Chart(trips_by_event).mark_bar().encode(
    x="daily_trips",
    y=alt.Y("event_day", title = ""),
    column=alt.Column("day_type:N", title = ""),
    row=alt.Row("schedule_name", title=""),
    color=alt.Color("schedule_name:N"), 
).properties(width=150, height=50)

In [ ]:
selection = alt.selection_point(fields=["schedule_name"], bind="legend")
alt.Chart(trips_by_event_wide).mark_bar().encode(
    x="change_daily_trips",
    y="route_name:N",
    column=alt.Column("day_type:N", title=""),
    color=alt.Color("schedule_name:N"), 
    opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.1)),
).add_params(selection)